In [3]:
import jax
import jax.numpy as jnp
import netket as nk
import netket.experimental as nkx
import numpy as np
from pyscf import gto, scf, fci
from flax import linen as nn
import flax.nnx as nnx
import optax
from tqdm import tqdm
from functools import partial
from jax import flatten_util
from sampler import  SingleStateAnsatz,mcmc_sampler,hi

model = SingleStateAnsatz(4,8,rngs=nnx.Rngs(21))

In [4]:
import orbax.checkpoint as ocp
from pathlib import Path
# 1. 获取当前工作目录的绝对路径
current_dir = Path.cwd().parent  # 或者用 Path(__file__).parent 如果在脚本中
ckpt_dir = current_dir / "暂存态"
# 2. 使用 PyTreeCheckpointer 而不是 StandardCheckpointer
checkpointer = ocp.PyTreeCheckpointer()

# 3. 保存模型（使用绝对路径）
ground_model = SingleStateAnsatz(4,12,rngs=nnx.Rngs(0))
graphdef, gs_state = nnx.split(ground_model)
save_path = str(ckpt_dir / "ground_state")  # 转为字符串
restore_state = checkpointer.restore(save_path, gs_state)
ground_model = nnx.merge(graphdef, restore_state)


In [5]:

def make_get_all_next_states(edges):
    # 把 edges 变成 静态 Python 元组！核心！
    edges = tuple(tuple(e) for e in edges)

    @jax.jit
    def get_all_next_states_jit(S: jnp.ndarray):
        s_arr = S
        next_states = []
        masks = []

        # 现在 edges 是 Python 静态元组 → 完全安全！
        for (i, j) in edges:
            occ_i = s_arr[i]
            occ_j = s_arr[j]
            valid = (occ_i == 1) & (occ_j == 0) | (occ_i == 0) & (occ_j == 1)
            new_state = s_arr.at[i].set(occ_j).at[j].set(occ_i)
            next_states.append(new_state)
            masks.append(valid)

        return jnp.stack(next_states), jnp.stack(masks)
    
    return get_all_next_states_jit

def make_metropolis_hastings_step(edges, log_pdf):
    get_all_next = make_get_all_next_states(edges)
    
    @jax.jit
    def metropolis_hastings_step_jit(S: jnp.ndarray, key: jax.Array):
        # 1. 候选
        candidates, valid_mask = get_all_next(S)
        key, subk = jax.random.split(key)
        idx = jax.random.choice(subk, candidates.shape[0])
        S_cand = candidates[idx]
        is_valid = valid_mask[idx]

        # ==============================
        # ✅ 严格正确：log_pdf = ln(ψ)
        # ==============================
        log_accept_ratio = 2 * (
            jnp.real(log_pdf(S_cand)) - jnp.real(log_pdf(S))
        )

        # 接受条件（无exp，超快）
        key, subk = jax.random.split(key)
        u = jax.random.uniform(subk)
        accept = is_valid & (log_accept_ratio > jnp.log(u))

        S_new = jnp.where(accept, S_cand, S)
        return S_new, accept, key

    return metropolis_hastings_step_jit
import jax
import jax.numpy as jnp
from functools import partial

# 🔥 正确静态参数
@partial(jax.jit, static_argnums=(0, 1, 3, 4,5))
def mcmc_sampler(
    n_samples: int,
    n_warmup: int,
    initial_state: jnp.ndarray,
    edges: tuple[tuple[int, int]],
    log_pdf: callable,
    seed: int = 42
):
    key = jax.random.PRNGKey(seed)
    mh_step = make_metropolis_hastings_step(edges, log_pdf)

    # 预烧
    def warmup_loop(carry, _):
        state, rng = carry
        state, _, rng = mh_step(state, rng)
        return (state, rng), None

    (current_state, key), _ = jax.lax.scan(
        warmup_loop,
        (initial_state, key),
        xs=None,
        length=n_warmup
    )

    # 采样
    def sample_loop(carry, _):
        state, rng = carry
        state, accepted, rng = mh_step(state, rng)
        return (state, rng), state

    (_, _), samples = jax.lax.scan(
        sample_loop,
        (current_state, key),
        xs=None,
        length=n_samples
    )

    return samples

In [14]:
samples = mcmc_sampler(n_samples=2000,n_warmup=200,
              initial_state=jnp.array([1,0,1,0]),
                                      edges=((0,1), (2,3)),
                                      log_pdf = ground_model,seed=12)


In [15]:
samples.shape

(2000, 4)

In [ ]:
single_ansatz(jnp.array([1,0,1,0]))

In [16]:
# ✅ 关键：把每一行变成一个可哈希的元组，才能统计频次
arr = samples
rows = [tuple(row) for row in np.asarray(arr)]

# 统计每一行出现多少次
from collections import Counter
count = Counter(rows)
count

Counter({(1, 0, 1, 0): 1572,
         (0, 1, 0, 1): 381,
         (1, 0, 0, 1): 29,
         (0, 1, 1, 0): 18})

In [17]:
log_psi = ground_model(hi.all_states())
jnp.exp(2*jnp.real(log_psi))/sum(jnp.exp(2*jnp.real(log_psi)))

Array([0.11544255, 0.00464553, 0.01122364, 0.86868828], dtype=float64)

In [18]:
hi.all_states()

Array([[0, 1, 0, 1],
       [0, 1, 1, 0],
       [1, 0, 0, 1],
       [1, 0, 1, 0]], dtype=int8)